In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('WELFake_Dataset.csv',index_col=0)
df

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
...,...,...,...
72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0
72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1
72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0
72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0


In [3]:
df.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72134 entries, 0 to 72133
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   71576 non-null  object
 1   text    72095 non-null  object
 2   label   72134 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.2+ MB


In [5]:
df.describe()

,label
count,72134.000000
mean,0.514404
std,0.499796
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [6]:
df=df.dropna(subset=['title'])
df['title']=df['title'].astype('str')
df=df.dropna(subset=['text'])
df['text']=df['text'].astype('str')
print(f"Null are droped in 'title' and 'text'")

Null are droped in 'title' and 'text'


In [7]:
import re
def clean_text(title):
    text = title.lower()
    text=re.sub(r"http\S+", "", text) 
    text=re.sub(r"[^a-z\s]", "", text)
    return text
df['title'] = df['title'].apply(clean_text)
print(df['title'][0])

law enforcement on high alert following threats against cops and whites on by blacklivesmatter and fyf terrorists video


In [8]:
df.head()

,title,text,label
0,law enforcement on high alert following threat...,No comment is expected from Barack Obama Membe...,1
2,unbelievable obamas attorney general says most...,"Now, most of the demonstrators gathered last ...",1
3,bobby jindal raised hindu uses story of christ...,A dozen politically active pastors came here f...,0
4,satan russia unvelis an image of its terrifyi...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
5,about time christian group sues amazon and spl...,All we can say on this one is it s about time ...,1


In [9]:
df.isnull().sum()

title    0
text     0
label    0
dtype: int64

In [10]:
df.columns


Index(['title', 'text', 'label'], dtype='object')

In [11]:
print(df['label'].value_counts())
print()
print('0 = Real news')
print('1 = Fake news')

label
1    36509
0    35028
Name: count, dtype: int64

0 = Real news
1 = Fake news


In [12]:
df['tokens']=df['title'].apply(lambda x:x.split() )
df[['title', 'text', 'tokens','label']].head()

,title,text,tokens,label
0,law enforcement on high alert following threat...,No comment is expected from Barack Obama Membe...,"[law, enforcement, on, high, alert, following,...",1
2,unbelievable obamas attorney general says most...,"Now, most of the demonstrators gathered last ...","[unbelievable, obamas, attorney, general, says...",1
3,bobby jindal raised hindu uses story of christ...,A dozen politically active pastors came here f...,"[bobby, jindal, raised, hindu, uses, story, of...",0
4,satan russia unvelis an image of its terrifyi...,"The RS-28 Sarmat missile, dubbed Satan 2, will...","[satan, russia, unvelis, an, image, of, its, t...",1
5,about time christian group sues amazon and spl...,All we can say on this one is it s about time ...,"[about, time, christian, group, sues, amazon, ...",1


In [59]:
fake_df=df[df['label']==1].sample(5000,random_state=42)
real_df=df[df['label']==0].sample(5000,random_state=42)
train_df = pd.concat([fake_df, real_df])
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(len(train_df))

10000


In [60]:
sentences = train_df['tokens'].tolist()
print(sentences[0])

['germany', 'france', 'warn', 'citizens', 'in', 'jerusalem', 'amid', 'planned', 'us', 'recognition']


In [61]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences   = sentences,
    vector_size = 100,    
    window      = 5,      
    min_count   = 2,      
    workers     = 4,
    epochs      = 10,
    seed        = 42
)

print('word2vec trained')
print('vocabulary size :', len(w2v_model.wv))

word2vec trained
vocabulary size : 7551


In [62]:
w2v_model.wv.most_similar('government', topn=20)

[('fight', 0.9988027215003967),
 ('philippines', 0.998718798160553),
 ('germany', 0.998640239238739),
 ('aid', 0.9986184239387512),
 ('rebels', 0.9986183643341064),
 ('bid', 0.998587429523468),
 ('funding', 0.9985840916633606),
 ('transgender', 0.9984614849090576),
 ('militant', 0.9983907341957092),
 ('reach', 0.9983507990837097),
 ('remains', 0.9983404278755188),
 ('summit', 0.9983253479003906),
 ('venezuela', 0.9983153939247131),
 ('international', 0.9983135461807251),
 ('pakistan', 0.9982926845550537),
 ('french', 0.9982529878616333),
 ('yemen', 0.9982438683509827),
 ('debt', 0.9982015490531921),
 ('rohingya', 0.9981955289840698),
 ('chinas', 0.9980998635292053)]

In [63]:
import numpy as np
def text_to_vector(text):
    tokens = clean_text(text).split()
    vectors = []
    for word in tokens:
        if word in w2v_model.wv:
            vectors.append(w2v_model.wv[word])
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)


test ='government admits secret conspiracy'
# test = ''
vec  = text_to_vector(test)
print('input  :', test)
print('output : vector of shape \n', vec)


input  : government admits secret conspiracy
output : vector of shape 
 [ 0.1504076   0.04034025 -0.2785713   0.24888426 -0.38306764  0.1773707
  0.27856576  0.34344974 -0.28955758 -0.11873311 -0.6610781   0.23965573
  0.05722769  0.59768474 -0.08090635 -0.3041397   0.04471256 -0.58285326
 -0.19195604 -0.06447816  0.15032533  0.28866965 -0.12987067  0.60830337
  0.37227598 -0.26165223 -0.48505512  0.06832742 -0.3022723  -0.15307723
 -0.03808303 -0.22352543  0.0374786   0.00329506  0.03884992 -0.10843888
  0.3714047  -0.66696095 -0.20827067  0.27654338  0.05661598 -0.05062462
  0.35656807 -0.18299857 -0.04215503 -0.12458112  0.27316552  0.47360367
  0.85440034 -0.37550792 -0.14929716 -0.8610142  -0.06696311 -0.15029626
  0.39653844 -0.01576209 -0.22133443  0.13634852 -0.14700548  0.1911881
 -0.04576443 -0.20541815 -0.42110994  0.21228711  0.3872013   0.24878737
  0.22663757 -0.29483673 -0.7650823  -0.11319578  0.06991526 -0.04216285
 -0.19427373 -0.39956036  0.29696083 -0.1917358   0.12

In [64]:
x=np.array(train_df['title'].apply(text_to_vector).tolist(),dtype=np.float32)
y=train_df['label'].values.astype(np.float32).reshape(-1, 1)

In [65]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print('train size :', x_train.shape[0])
print('test  size :', x_test.shape[0])

train size : 8000
test  size : 2000


In [66]:
import torch
import torch.nn as nn
import torch.optim as optim

x_train_t = torch.tensor(x_train)
y_train_t = torch.tensor(y_train)
x_test_t  = torch.tensor(x_test)
y_test_t  = torch.tensor(y_test)


In [67]:
fakenewsmodel=nn.Sequential(
    nn.Linear(100, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    # nn.Dropout(0.3),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1),
    nn.Sigmoid()
    
)

model = fakenewsmodel
print(model)

Sequential(
  (0): Linear(in_features=100, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=32, bias=True)
  (6): ReLU()
  (7): Dropout(p=0.3, inplace=False)
  (8): Linear(in_features=32, out_features=1, bias=True)
  (9): Sigmoid()
)


In [68]:
loss_fn   = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [69]:
from torch.utils.data import DataLoader, TensorDataset

dataset    = TensorDataset(x_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
EPOCHS = 30
losses = []
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    correct    = 0
    total      = 0
    for xb, yb in dataloader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += ((pred > 0.5).float() == yb).sum().item()
        total      += len(yb)
    avg_loss = total_loss / len(dataloader)
    acc      = correct / total * 100
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1:2d}/{EPOCHS}  loss: {avg_loss:.4f}  accuracy: {acc:.1f}%')
model.eval()
print('training done')

epoch  5/30  loss: 0.4867  accuracy: 77.7%
epoch 10/30  loss: 0.4715  accuracy: 78.2%
epoch 15/30  loss: 0.4634  accuracy: 79.2%
epoch 20/30  loss: 0.4566  accuracy: 79.0%
epoch 25/30  loss: 0.4585  accuracy: 79.5%
epoch 30/30  loss: 0.4530  accuracy: 79.7%
training done


In [70]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    preds     = model(x_test_t)
    predicted = (preds > 0.5).float()
y_true = y_test_t.numpy().flatten().astype(int)
y_pred = predicted.numpy().flatten().astype(int)

print('accuracy :', round(accuracy_score(y_true, y_pred) * 100, 2), '%')
print()
# print('accuracy :', round(classification_report(y_true, y_pred) * 100, 2), '%')

accuracy : 78.75 %



In [71]:
def predict(text):
    vec    = text_to_vector(text)
    tensor = torch.tensor([vec], dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        out = model(tensor)
    fake_prob = float(out[0][0])
    real_prob = 1 - fake_prob
    label     = 'FAKE' if fake_prob > 0.5 else 'REAL'
    print('text       :', text[:80])
    print('prediction :', label)
    print('fake prob  :', round(fake_prob * 100, 2), '%')
    print('real prob  :', round(real_prob * 100, 2), '%')
    print()
    return label, fake_prob, real_prob